# scGPT MAPT Tangle Stage — Preprocessing & EDA

**Project:** Fine-tuning scGPT to predict neurofibrillary tangle stage from single-nucleus RNA-seq data  
**Dataset:** Morabito et al. 2021 (GSE174367) — Single-nucleus transcriptomic characterization of Alzheimer's disease  
**Output:** `neurons_hvg_preprocessed.h5ad` — preprocessed neuron subset ready for scGPT fine-tuning

---
**Pipeline overview:**
1. Load raw 10x expression matrix and cell metadata
2. Explore dataset composition and target variable distribution
3. Subset to neurons (biological decision)
4. Analyze MAPT expression across tangle stages
5. Quality control, normalization, HVG selection
6. Baseline UMAP to motivate deep learning approach
7. Save preprocessed data

## 1. Setup and Imports

**Environment:** Run using the `scgpt_mapt` conda environment on NEU HPC.  
```bash
conda activate scgpt_mapt
```
This notebook does not require scGPT or PyTorch — only scanpy and standard data science libraries.

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for HPC
import matplotlib.pyplot as plt
import seaborn as sns
from importlib.metadata import version

# Set working directory to project folder
os.chdir('/scratch/arumuganainar.d/projects/scgpt')

# Suppress scanpy verbosity for cleaner output
sc.settings.verbosity = 1

print('Scanpy version:', version('scanpy'))
print('Working directory:', os.getcwd())

## 2. Data Loading

**Source:** GEO accession GSE174367 (Morabito et al. 2021, *Nature Genetics*)  
**Files:**
- `GSE174367_snRNA-seq_filtered_feature_bc_matrix.h5` — 10x formatted gene expression matrix (261MB)
- `GSE174367_snRNA-seq_cell_meta.csv.gz` — cell-level metadata including tangle stage, diagnosis, cell type

Barcodes are aligned between the two files using their shared index.

In [ ]:
def load_dataset(h5_path, meta_path):
    """
    Load 10x expression matrix and attach cell metadata.
    
    Parameters
    ----------
    h5_path : str
        Path to 10x HDF5 expression matrix
    meta_path : str
        Path to cell metadata CSV (gzipped)
    
    Returns
    -------
    adata : AnnData
        Expression matrix with metadata attached to obs
    """
    # Load expression matrix
    adata = sc.read_10x_h5(h5_path)
    adata.var_names_make_unique()  # resolve duplicate gene names
    
    # Load metadata
    meta = pd.read_csv(meta_path, index_col=0)
    
    # Align barcodes — only keep cells present in both files
    common_barcodes = adata.obs_names.intersection(meta.index)
    adata = adata[common_barcodes].copy()
    meta_aligned = meta.loc[common_barcodes]
    
    # Attach metadata columns to adata.obs
    for col in ['Diagnosis', 'Cell.Type', 'Tangle.Stage', 'Age', 'Sex', 'Batch']:
        adata.obs[col] = meta_aligned[col].values
    
    return adata, meta_aligned


adata, meta = load_dataset(
    'GSE174367_snRNA-seq_filtered_feature_bc_matrix.h5',
    'GSE174367_snRNA-seq_cell_meta.csv.gz'
)

print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print(f'Metadata columns: {meta.columns.tolist()}')

## 3. Dataset Exploration

Before any filtering, we examine the full dataset composition to understand:
- What cell types are present
- How tangle stages are distributed
- Whether there are missing values in key columns
- The balance between AD and control samples

**Key metadata columns:**
- `Tangle.Stage` — neurofibrillary tangle staging (our prediction target)
- `Cell.Type` — cell type annotation (ODC, EX, INH, ASC, MG, OPC)
- `Diagnosis` — AD vs Control
- `PMI` — post-mortem interval (technical covariate, not used as feature)

In [ ]:
def explore_dataset(adata):
    """
    Print key dataset statistics for quality assessment.
    
    Parameters
    ----------
    adata : AnnData
        Dataset with metadata attached
    """
    print('=== Cell Type Distribution ===')
    print(adata.obs['Cell.Type'].value_counts())
    
    print('\n=== Tangle Stage Distribution ===')
    print(adata.obs['Tangle.Stage'].value_counts().sort_index())
    
    print('\n=== Diagnosis Distribution ===')
    print(adata.obs['Diagnosis'].value_counts())
    
    print('\n=== Missing Values ===')
    print(adata.obs[['Tangle.Stage', 'Cell.Type', 'Diagnosis', 'PMI']].isnull().sum())


explore_dataset(adata)

**Observations:**
- Oligodendrocytes (ODC) dominate at ~60% of cells — expected in postmortem brain tissue
- Tangle stages 3 and 4 are absent from this dataset — a known limitation
- Stage 6 is heavily overrepresented (~52%) reflecting postmortem brain bank recruitment bias toward late-stage patients
- PMI has ~4,575 missing values — excluded as a feature
- Tangle Stage has zero missing values — clean prediction target

## 4. Neuron Subsetting

**Biological justification:** Neurofibrillary tau tangles composed of hyperphosphorylated MAPT protein form **primarily in neurons**, not in glial cells. Analyzing all cell types together would dilute the tau-related transcriptional signal with irrelevant glial biology.

We retain:
- **EX** — excitatory neurons (glutamatergic)
- **INH** — inhibitory neurons (GABAergic)

We exclude: ODC (oligodendrocytes), ASC (astrocytes), MG (microglia), OPC (oligodendrocyte precursors), PER.END (pericytes/endothelial)

In [ ]:
def subset_neurons(adata):
    """
    Subset AnnData to excitatory and inhibitory neurons only.
    
    Tau tangles form primarily in neurons — including glial cells
    would dilute disease-relevant transcriptional signal.
    
    Parameters
    ----------
    adata : AnnData
        Full dataset with all cell types
    
    Returns
    -------
    neurons : AnnData
        Subset containing only EX and INH neurons
    """
    neurons = adata[adata.obs['Cell.Type'].isin(['EX', 'INH'])].copy()
    return neurons


neurons = subset_neurons(adata)

print(f'Total cells: {adata.n_obs:,}')
print(f'Neurons only: {neurons.n_obs:,} ({neurons.n_obs/adata.n_obs*100:.1f}% of total)')
print(f'\nNeuron type breakdown:')
print(neurons.obs['Cell.Type'].value_counts())
print(f'\nTangle stage distribution in neurons:')
print(neurons.obs['Tangle.Stage'].value_counts().sort_index())
print(f'\nMAPT present in dataset: {"MAPT" in neurons.var_names}')

## 5. MAPT Expression Analysis Across Tangle Stages

Before preprocessing, we examine MAPT expression across tangle stages in raw log-normalized counts to understand the biological signal we are trying to model.

**Expected:** MAPT expression should increase with tangle stage if tau transcription drives pathology.  
**Actual finding:** MAPT peaks at Stage 2 then declines at Stages 5 and 6 — a counter-intuitive result with an important biological explanation.

In [ ]:
def analyze_mapt_expression(neurons):
    """
    Quantify and visualize MAPT expression across tangle stages.
    
    Parameters
    ----------
    neurons : AnnData
        Neuron subset with Tangle.Stage metadata
    
    Returns
    -------
    neurons : AnnData
        Same object with MAPT_expr column added to obs
    """
    # Extract MAPT expression vector
    mapt_expr = neurons[:, 'MAPT'].X.toarray().flatten()
    neurons.obs['MAPT_expr'] = mapt_expr
    
    # Print summary statistics per stage
    print('=== MAPT Expression by Tangle Stage ===')
    for stage in ['Stage 1', 'Stage 2', 'Stage 5', 'Stage 6']:
        mask = neurons.obs['Tangle.Stage'] == stage
        expr = neurons.obs.loc[mask, 'MAPT_expr']
        print(f'{stage}: mean={expr.mean():.3f}, '
              f'% expressing={(expr > 0).mean()*100:.1f}%')
    
    # Boxplot
    fig, ax = plt.subplots(figsize=(8, 5))
    neurons.obs.boxplot(column='MAPT_expr', by='Tangle.Stage', ax=ax)
    ax.set_title('MAPT Expression by Tangle Stage (Neurons)')
    ax.set_xlabel('Tangle Stage')
    ax.set_ylabel('MAPT Expression (raw counts)')
    plt.suptitle('')  # remove default boxplot title
    plt.tight_layout()
    plt.savefig('figures/MAPT_by_tangle_stage.png', dpi=150)
    print('\nFigure saved: figures/MAPT_by_tangle_stage.png')
    
    return neurons


os.makedirs('figures', exist_ok=True)
neurons = analyze_mapt_expression(neurons)

**Biological interpretation — Survivor Bias Hypothesis:**

MAPT expression peaks at Stage 2 (mean=11.5, 92.3% expressing) then declines at Stages 5 and 6. This is consistent with **survivor bias in neurodegeneration**:

- Neurons with the highest MAPT expression are most vulnerable to tau aggregation
- These vulnerable neurons die first as disease progresses
- By Stage 5/6, only the most resilient (lower MAPT) neurons survive to be sequenced
- Therefore late-stage samples are enriched for MAPT-low survivors

**Implication for modeling:** A simple MAPT expression threshold cannot distinguish tangle stages. The model must learn the broader transcriptional context — co-expressed genes, pathway activity, cell state — rather than relying on MAPT alone. This motivates the use of scGPT which captures gene-gene relationships through attention mechanisms.

## 6. Quality Control and Preprocessing

Standard single-cell preprocessing pipeline:

1. **QC filtering** — remove low-quality cells and rarely expressed genes
2. **Normalization** — scale all cells to 10,000 total counts to remove sequencing depth effects
3. **Log transformation** — compress dynamic range, approximate normal distribution
4. **HVG selection** — retain 3,000 most variable genes to reduce noise
5. **Force MAPT inclusion** — MAPT is not highly variable (expressed consistently across neurons) but is our primary gene of interest, so we manually include it
6. **Tangle stage encoding** — map stage labels to integers for model training

In [ ]:
def preprocess_neurons(neurons, n_hvg=3000):
    """
    Standard single-cell preprocessing pipeline for neuron subset.
    
    Parameters
    ----------
    neurons : AnnData
        Raw neuron subset
    n_hvg : int
        Number of highly variable genes to select (default 3000)
    
    Returns
    -------
    neurons_hvg : AnnData
        Preprocessed neurons subset to HVG + MAPT
    """
    # Step 1 — QC filtering
    # Min 200 genes per cell removes empty droplets and dead cells
    # Min 10 cells per gene removes extremely rare transcripts
    sc.pp.filter_cells(neurons, min_genes=200)
    sc.pp.filter_genes(neurons, min_cells=10)
    print(f'After QC: {neurons.n_obs:,} cells x {neurons.n_vars:,} genes')
    
    # Step 2 — Normalize total counts per cell to 10,000
    # Removes technical variation from sequencing depth differences
    sc.pp.normalize_total(neurons, target_sum=1e4)
    
    # Step 3 — Log1p transformation
    # Stabilizes variance and compresses dynamic range
    sc.pp.log1p(neurons)
    
    # Step 4 — Highly variable gene selection
    # Retains genes that vary most across cells — most biologically informative
    sc.pp.highly_variable_genes(neurons, n_top_genes=n_hvg)
    print(f'Highly variable genes selected: {neurons.var.highly_variable.sum()}')
    
    # Step 5 — Force MAPT into HVG set
    # MAPT is expressed consistently across neurons (83-92% expressing)
    # so it does not rank as highly variable by variance alone.
    # We manually include it because it is the primary gene of interest.
    neurons.var.loc['MAPT', 'highly_variable'] = True
    print(f'HVG count after forcing MAPT inclusion: {neurons.var.highly_variable.sum()}')
    print(f'MAPT in HVG set: {neurons.var.loc["MAPT", "highly_variable"]}')
    
    # Step 6 — Subset to HVG only
    neurons_hvg = neurons[:, neurons.var.highly_variable].copy()
    print(f'Final matrix: {neurons_hvg.n_obs:,} cells x {neurons_hvg.n_vars:,} genes')
    
    # Step 7 — Encode tangle stage as numeric labels for model training
    # Stages 3 and 4 are absent from this dataset
    stage_mapping = {'Stage 1': 0, 'Stage 2': 1, 'Stage 5': 2, 'Stage 6': 3}
    neurons_hvg.obs['tangle_label'] = neurons_hvg.obs['Tangle.Stage'].map(stage_mapping)
    print(f'\nLabel distribution:')
    print(neurons_hvg.obs['tangle_label'].value_counts().sort_index())
    
    return neurons_hvg


neurons_hvg = preprocess_neurons(neurons)

**Preprocessing decisions:**
- `min_genes=200` — standard threshold; removes empty droplets and severely damaged cells
- `target_sum=1e4` — 10,000 counts is the field standard for single-cell normalization
- `n_hvg=3000` — larger than typical (2000) to retain more disease-relevant variation given the subtle signal between tangle stages
- MAPT forced in — scientifically motivated override of the automated filter

## 7. Baseline UMAP — Motivating the Deep Learning Approach

Before fine-tuning scGPT, we establish a baseline using standard dimensionality reduction (PCA → UMAP). This serves two purposes:

1. **Quality check** — confirm the data has biological structure (cell types should separate)
2. **Motivation** — demonstrate that tangle stages do NOT separate with basic methods, justifying the use of a foundation model

We visualize three properties simultaneously: tangle stage, MAPT expression, and neuron type.

In [ ]:
def compute_and_plot_baseline_umap(neurons_hvg):
    """
    Compute PCA and UMAP embeddings and generate three-panel baseline figure.
    
    Three panels:
    - Tangle Stage: tests if disease severity separates in PCA space
    - MAPT Expression: tests if MAPT alone drives transcriptional structure
    - Neuron Type: validates UMAP quality (EX/INH should separate cleanly)
    
    Parameters
    ----------
    neurons_hvg : AnnData
        Preprocessed neuron subset with HVG selected
    
    Returns
    -------
    neurons_hvg : AnnData
        Same object with PCA and UMAP coordinates stored in obsm
    """
    # Scale before PCA — centers and scales each gene
    sc.pp.scale(neurons_hvg, max_value=10)
    
    # PCA — reduce to 50 principal components
    sc.pp.pca(neurons_hvg, n_comps=50)
    print('PCA complete')
    
    # Neighbor graph using top 40 PCs
    sc.pp.neighbors(neurons_hvg, n_neighbors=15, n_pcs=40)
    
    # UMAP projection
    sc.tl.umap(neurons_hvg)
    print('UMAP complete')
    
    # Three-panel figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sc.pl.umap(neurons_hvg, color='Tangle.Stage',
               ax=axes[0], show=False, title='Tangle Stage')
    
    sc.pl.umap(neurons_hvg, color='MAPT',
               ax=axes[1], show=False,
               title='MAPT Expression', color_map='Reds')
    
    sc.pl.umap(neurons_hvg, color='Cell.Type',
               ax=axes[2], show=False, title='Neuron Type')
    
    plt.tight_layout()
    plt.savefig('figures/UMAP_neurons_baseline.png', dpi=150, bbox_inches='tight')
    print('Figure saved: figures/UMAP_neurons_baseline.png')
    
    return neurons_hvg


neurons_hvg = compute_and_plot_baseline_umap(neurons_hvg)

**Baseline UMAP Interpretation:**

- **Tangle Stage (left):** All four stages are completely intermixed — PCA cannot separate disease severity. This is the core problem this project addresses.
- **MAPT Expression (center):** Uniform expression across the UMAP — MAPT alone does not drive transcriptional structure, consistent with the survivor bias finding above.
- **Neuron Type (right):** EX and INH neurons form clean, distinct clusters — confirms the UMAP is working correctly and can detect strong biological signal.

**Conclusion:** Standard PCA-based dimensionality reduction fails to separate tangle stages because the transcriptional differences between stages are subtle and distributed across many genes simultaneously. A transformer-based foundation model (scGPT) that captures gene-gene co-expression relationships is needed to detect this signal. This motivates the fine-tuning approach in `scgpt_finetune.ipynb`.

## 8. Save Preprocessed Data

Save the preprocessed neuron subset as an AnnData HDF5 file. This file is the direct input to `scgpt_finetune.ipynb` — no reprocessing needed.

**What is saved:**
- Expression matrix: 12,331 neurons × 3,001 genes (3,000 HVG + MAPT)
- Metadata: Tangle.Stage, tangle_label, Cell.Type, Diagnosis, Age, Sex, Batch, MAPT_expr
- Embeddings: PCA (50 components), UMAP (2 components)
- Gene metadata: HVG flags, dispersion statistics

In [ ]:
def save_preprocessed_data(neurons_hvg, output_path='neurons_hvg_preprocessed.h5ad'):
    """
    Save preprocessed AnnData object for downstream fine-tuning.
    
    Parameters
    ----------
    neurons_hvg : AnnData
        Fully preprocessed neuron subset
    output_path : str
        Path to save the h5ad file
    """
    neurons_hvg.write_h5ad(output_path)
    
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    
    print(f'Saved: {output_path} ({file_size_mb:.1f} MB)')
    print(f'\nContents:')
    print(f'  Cells: {neurons_hvg.n_obs:,}')
    print(f'  Genes: {neurons_hvg.n_vars:,}')
    print(f'  Metadata columns: {neurons_hvg.obs.columns.tolist()}')
    print(f'  Embeddings: {list(neurons_hvg.obsm.keys())}')
    print(f'\nLabel distribution:')
    stage_map = {0: "Stage 1", 1: "Stage 2", 2: "Stage 5", 3: "Stage 6"}
    for label, count in neurons_hvg.obs['tangle_label'].value_counts().sort_index().items():
        print(f'  {stage_map[label]} (label {label}): {count:,} cells')
    print(f'\nNext step: run scgpt_finetune.ipynb')


save_preprocessed_data(neurons_hvg)